In [2]:
import os
import polars as pl
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error,
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    roc_auc_score
)

# ==========================================
# 0. 基础配置
# ==========================================
INPUT_DIR = "./processed_data_light" # 之前分批保存的文件夹路径
ACTIVATION_THRESHOLD = 0.01
target = "current_threat"

features = [
    "norm_origin_pos_x", "norm_origin_pos_y", 
    "defensive_hull_area", "is_ball_inside_def_hull", 
    "pc_5m_lf", "pc_5m_rf", "pc_5m_lb", "pc_5m_rb", 
    "pc_15m_lf", "pc_15m_rf", "pc_15m_lb", "pc_15m_rb", 
    "pos_formation_height", "pos_formation_width", 
    "def_formation_height", "def_formation_width", 
    "pos_space_0_area", "pos_space_1_area", "pos_space_2_area", "pos_space_3_area", 
    "def_space_0_area", "def_space_1_area", "def_space_2_area", "def_space_3_area", 
    "pc_ratio_def_space_0", "pc_ratio_def_space_1", "pc_ratio_def_space_2", "pc_ratio_def_space_3",
    "dist_to_goal", "angle_to_goal"
]

# ==========================================
# 1. 流式读取与内存安全过滤 (Polars Lazy API)
# ==========================================
print("⏳ [Stage 0] 正在通过 Lazy API 扫描并过滤所有比赛特征集...")

# 1. scan_parquet 只读取元数据，不占用真实内存
lazy_df = pl.scan_parquet(f"{INPUT_DIR}/*.parquet")

# 2. 在落盘状态下直接过滤和选择列（极大地节省内存）
filtered_lazy_df = (
    lazy_df
    .filter(pl.col(target) > 0)
    .select(features + [target])
)

# 3. collect() 正式触发计算，将精华数据拉入内存
df_filtered = filtered_lazy_df.collect()

print(f"✅ 数据抽取完成！共计 {df_filtered.shape[0]} 行高价值样本进入机器学习管线。")

# 转化为 Pandas 供 Scikit-Learn 与 LightGBM 使用
df_ml = df_filtered.to_pandas()
df_ml["is_ball_inside_def_hull"] = df_ml["is_ball_inside_def_hull"].astype(int)
df_ml = df_ml.fillna(0)

# 生成二分类标签
df_ml["is_active"] = (df_ml[target] > ACTIVATION_THRESHOLD).astype(int)

# 划分全局训练/测试集
X = df_ml[features]
y_class = df_ml["is_active"]
y_reg = df_ml[target]

X_train, X_test, y_train_class, y_test_class, y_train_reg, y_test_reg = train_test_split(
    X, y_class, y_reg, test_size=0.2, random_state=42
)

# ==========================================
# 2. 阶段一：概率探测器 (LGBM Classifier)
# ==========================================
print("🔥 [Stage 1] 正在训练进攻激活概率模型 (Classifier)...")

num_neg = (y_train_class == 0).sum()
num_pos = (y_train_class == 1).sum()
scale_weight = num_neg / num_pos if num_pos > 0 else 1.0

prob_model = lgb.LGBMClassifier(
    n_estimators=400,
    learning_rate=0.03,      
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_weight,
    random_state=42,
    n_jobs=-1,
    verbose=-1 # 隐藏啰嗦的构建日志
)

prob_model.fit(X_train, y_train_class)

# ==========================================
# 3. 阶段二：威胁度回归器 (LGBM Regressor)
# ==========================================
print("🔥 [Stage 2] 正在训练连续威胁度模型 (Regressor)...")

active_train_mask = (y_train_class == 1)
X_train_active = X_train[active_train_mask]
y_train_reg_active = y_train_reg[active_train_mask]

lgb_model = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

lgb_model.fit(X_train_active, y_train_reg_active)

# ==========================================
# 4. 组合模型：生成 P * Expected_Threat
# ==========================================
print("🔥 [Stage 3] 正在组合双阶段模型并计算最终预期威胁...")

p_active_pred = prob_model.predict_proba(X)[:, 1]
expected_threat_pred = lgb_model.predict(X)

df_ml["p_active"] = p_active_pred
df_ml["expected_threat_if_active"] = expected_threat_pred
df_ml["p_expected_threat"] = p_active_pred * expected_threat_pred

print("✅ 计算完成！结果已储存至 df_ml['p_expected_threat']")

test_final_pred = df_ml.loc[X_test.index, "p_expected_threat"]
final_rmse = np.sqrt(mean_squared_error(y_test_reg, test_final_pred))
print(f"\n=== 组合模型 (Hurdle Model) 测试集表现 ===")
print(f"Final RMSE: {final_rmse:.4f}")


# ==========================================
# 4. 组合模型：针对【测试集】生成预测
# ==========================================
print("🔥 [Stage 3] 正在对测试集进行双阶段模型推理...")

# 🚨 核心修改点：将 X 替换为 X_test 🚨
p_active_test_pred = prob_model.predict_proba(X_test)[:, 1]
expected_threat_test_pred = lgb_model.predict(X_test)

# 计算测试集最终的综合预期威胁 (P * E)
test_final_pred = p_active_test_pred * expected_threat_test_pred


# ==========================================
# 5. 阶段一检验：二分类模型 (is_active) 性能评估
# ==========================================
print("\n" + "="*40)
print("📊 [Stage 1 Evaluation] 激活概率分类器性能检验")
print("="*40)

# 🚨 核心修改点：使用针对测试集的预测概率进行硬判定
y_test_class_pred = (p_active_test_pred >= 0.5).astype(int)

# 此时两边的样本量都是 81,786，完美对齐
acc = accuracy_score(y_test_class, y_test_class_pred)
precision = precision_score(y_test_class, y_test_class_pred)
recall = recall_score(y_test_class, y_test_class_pred)
f1 = f1_score(y_test_class, y_test_class_pred)
roc_auc = roc_auc_score(y_test_class, p_active_test_pred) # 输入测试集概率

print(f"准确率 (Accuracy) : {acc:.4f}")
print(f"精确率 (Precision): {precision:.4f}")
print(f"召回率 (Recall)   : {recall:.4f}")
print(f"F1-Score         : {f1:.4f}")
print(f"ROC-AUC 面积     : {roc_auc:.4f}")


# ==========================================
# 6. 阶段二检验：按真实威胁大小的分层 RMSE 评估
# ==========================================
print("\n" + "="*40)
print("🎯 [Stage 2 Evaluation] 分层威胁度 RMSE 检验")
print("="*40)

# 此时 y_test_reg.values 和 test_final_pred 的长度也是 81,786，不会再报错
df_eval = pd.DataFrame({
    'true_threat': y_test_reg.values,
    'pred_threat': test_final_pred
})

bins = [0, 0.01, 0.05, 0.15, 1.0]
labels = [
    "未激活区间 (0.00 - 0.01)", 
    "低度威胁区间 (0.01 - 0.05)", 
    "中度威胁区间 (0.05 - 0.15)", 
    "高危绝佳机会 (0.15+)"
]

df_eval['threat_strata'] = pd.cut(df_eval['true_threat'], bins=bins, labels=labels, right=False)

global_rmse = np.sqrt(mean_squared_error(df_eval['true_threat'], df_eval['pred_threat']))
print(f"🌍 整体测试集 RMSE: {global_rmse:.5f}\n")

print("🔍 分层检验详情:")
print(f"{'威胁度层级 (Strata)':<30} | {'样本量 (N)':<10} | {'分层 RMSE':<10}")
print("-" * 55)

for strata_name in labels:
    subset = df_eval[df_eval['threat_strata'] == strata_name]
    if len(subset) == 0:
        print(f"{strata_name:<30} | {'0':<10} | {'N/A':<10}")
        continue
    strata_rmse = np.sqrt(mean_squared_error(subset['true_threat'], subset['pred_threat']))
    print(f"{strata_name:<30} | {len(subset):<10} | {strata_rmse:.5f}")



⏳ [Stage 0] 正在通过 Lazy API 扫描并过滤所有比赛特征集...
✅ 数据抽取完成！共计 408926 行高价值样本进入机器学习管线。
🔥 [Stage 1] 正在训练进攻激活概率模型 (Classifier)...
🔥 [Stage 2] 正在训练连续威胁度模型 (Regressor)...
🔥 [Stage 3] 正在组合双阶段模型并计算最终预期威胁...
✅ 计算完成！结果已储存至 df_ml['p_expected_threat']

=== 组合模型 (Hurdle Model) 测试集表现 ===
Final RMSE: 0.0166
🔥 [Stage 3] 正在对测试集进行双阶段模型推理...

📊 [Stage 1 Evaluation] 激活概率分类器性能检验
准确率 (Accuracy) : 0.9094
精确率 (Precision): 0.9375
召回率 (Recall)   : 0.8892
F1-Score         : 0.9127
ROC-AUC 面积     : 0.9661

🎯 [Stage 2 Evaluation] 分层威胁度 RMSE 检验
🌍 整体测试集 RMSE: 0.01662

🔍 分层检验详情:
威胁度层级 (Strata)                 | 样本量 (N)    | 分层 RMSE   
-------------------------------------------------------
未激活区间 (0.00 - 0.01)            | 38217      | 0.00459
低度威胁区间 (0.01 - 0.05)           | 35152      | 0.01138
中度威胁区间 (0.05 - 0.15)           | 8080       | 0.03103
高危绝佳机会 (0.15+)                 | 337        | 0.16754


In [ ]:
test_final_pred

array([0.0242677 , 0.00124266, 0.0014937 , ..., 0.00046535, 0.00056897,
       0.03355061], shape=(81786,))

In [ ]:
"""
import joblib
import os

MODEL_DIR = "./saved_models"
os.makedirs(MODEL_DIR, exist_ok=True)

# 保存分类器和回归器
joblib.dump(prob_model, f"{MODEL_DIR}/prob_model_hurdle.pkl")
joblib.dump(lgb_model, f"{MODEL_DIR}/expected_threat_reg_hurdle.pkl")

print("✅ 模型已成功序列化并保存至本地。")
"""

'\nimport joblib\nimport os\n\nMODEL_DIR = "./saved_models"\nos.makedirs(MODEL_DIR, exist_ok=True)\n\n# 保存分类器和回归器\njoblib.dump(prob_model, f"{MODEL_DIR}/prob_model_hurdle.pkl")\njoblib.dump(lgb_model, f"{MODEL_DIR}/expected_threat_reg_hurdle.pkl")\n\nprint("✅ 模型已成功序列化并保存至本地。")\n'

In [ ]:
import glob
import os
import numpy as np
import polars as pl
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, classification_report, log_loss
import matplotlib.pyplot as plt

# ==============================================================================
# 1. 数据流式读取与标签预处理
# ==============================================================================
def load_and_prepare_data(data_dir: str):
    print("📂 开始流式读取 Parquet 批次文件...")
    file_pattern = os.path.join(data_dir, "processed_batch_*.parquet")
    files = glob.glob(file_pattern)
    
    if not files:
        raise FileNotFoundError(f"❌ 在 {data_dir} 未找到处理好的 Parquet 文件，请先运行特征工程脚本。")
        
    # 使用 Polars 惰性加载（Lazy），即使数据量极大也不会撑爆内存
    lazy_df = pl.scan_parquet(file_pattern)
    
    # 过滤掉 success 既不是 Exitoso 也不是 Fallido 的异常行（如有）
    lazy_df = lazy_df.filter(pl.col("success").is_in(["Exitoso", "Fallido"]))
    
    # 将目标变量映射为二分类标签：Exitoso (成功) -> 1, Fallido (失败) -> 0
    lazy_df = lazy_df.with_columns(
        pl.when(pl.col("success") == "Exitoso").then(1).otherwise(0).alias("target")
    )
    
    # 显式采集数据（Compute）
    df = lazy_df.collect()
    print(f"✅ 数据加载完成！样本总量: {df.shape[0]} 行, 特征总量: {df.shape[1]} 列。")
    return df

# ==============================================================================
# 2. 战术特征选择 (精选高价值标量，避开 7140 维的稠密网格列表)
# ==============================================================================
FEATURES = [
    # 传球基本属性
    "pass_length", 
    
    # 目标接球区域控制力 (你刚新增的杀手级特征)
    "dest_pc_5m", 
    
    # 动态几何与空间控制特征
    "path_pc_min_1", "path_pc_min_2", "path_pc_min_3", 
    
    # 防守方凸包特征
    #"is_ball_inside_def_hull",
]

# ==============================================================================
# 3. 按 Match ID 划分训练集与测试集 (防数据泄漏)
# ==============================================================================
def split_by_match(df: pl.DataFrame, train_ratio: float = 0.8):
    unique_matches = df["match_id"].unique().to_list()
    np.random.seed(42)  # 保证实验可重复
    np.random.shuffle(unique_matches)
    
    split_idx = int(len(unique_matches) * train_ratio)
    train_matches = unique_matches[:split_idx]
    test_matches = unique_matches[split_idx:]
    
    train_df = df.filter(pl.col("match_id").is_in(train_matches))
    test_df = df.filter(pl.col("match_id").is_in(test_matches))
    
    print(f"📊 战术切分完成：")
    print(f"   - 训练集: {len(train_matches)} 场比赛，共 {train_df.shape[0]} 个事件")
    print(f"   - 测试集: {len(test_matches)} 场比赛，共 {test_df.shape[0]} 个事件")
    
    return train_df, test_df

# ==============================================================================
# 4. 模型训练与评估
# ==============================================================================
def train_success_predictor(train_df: pl.DataFrame, test_df: pl.DataFrame):
    # 转换为 Numpy 供 LightGBM 消费
    X_train = train_df.select(FEATURES).to_numpy().astype(np.float32)
    y_train = train_df["target"].to_numpy().astype(np.int32)
    
    X_test = test_df.select(FEATURES).to_numpy().astype(np.float32)
    y_test = test_df["target"].to_numpy().astype(np.int32)
    
    # 构建 LightGBM 数据集
    train_data = lgb.Dataset(X_train, label=y_train, feature_name=FEATURES)
    test_data = lgb.Dataset(X_test, label=y_test, feature_name=FEATURES, reference=train_data)
    
    # 针对不平衡样本或二分类设置标准参数
    params = {
        "objective": "binary",
        "metric": ["binary_logloss", "auc"],
        "boosting_type": "gbdt",
        "learning_rate": 0.05,
        "num_leaves": 31,
        "max_depth": 6,
        "feature_fraction": 0.8,
        "verbose": -1,
        "seed": 42
    }
    
    print("\n🚀 开始训练 LightGBM 成功率预测模型...")
    model = lgb.train(
        params,
        train_data,
        num_boost_round=1000,
        valid_sets=[train_data, test_data],
        callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(50)]
    )
    
    # 预测并评估
    y_pred_proba = model.predict(X_test, num_iteration=model.best_iteration)
    y_pred_class = (y_pred_proba >= 0.5).astype(int)
    
    print("\n================ 模型评估报告 ================")
    print(f"📈 Test ROC-AUC : {roc_auc_score(y_test, y_pred_proba):.4f}")
    print(f"📉 Test LogLoss : {log_loss(y_test, y_pred_proba):.4f}")
    print("\n详细分类指标:")
    print(classification_report(y_test, y_pred_class, target_names=["Fallido (0)", "Exitoso (1)"]))
    
    # 输出特征重要性 (体育战术分析的核心)
    importance = model.feature_importance(importance_type="gain")
    idx = np.argsort(importance)[::-1]
    
    print("\n🎯 战术特征贡献度排行 (Top 10):")
    for i in range(min(10, len(FEATURES))):
        print(f"排第 {i+1}: {FEATURES[idx[i]]:<25} -> 增益贡献: {importance[idx[i]]:.2f}")
        
    return model

# ==============================================================================
# 主执行入口
# ==============================================================================
if __name__ == "__main__":
    # 假设你的 Parquet 文件存储在当前目录下的 processed_data 文件夹内
    DATA_DIRECTORY = "./processed_data_light" 
    
    # 1. 载入并清洗
    full_dataset = load_and_prepare_data(DATA_DIRECTORY)
    
    # 2. 阵营隔离切分
    train_set, test_set = split_by_match(full_dataset, train_ratio=0.8)
    
    # 3. 预测成功率
    bst_model = train_success_predictor(train_set, test_set)

📂 开始流式读取 Parquet 批次文件...
✅ 数据加载完成！样本总量: 643139 行, 特征总量: 63 列。
📊 战术切分完成：
   - 训练集: 303 场比赛，共 512503 个事件
   - 测试集: 76 场比赛，共 130636 个事件

🚀 开始训练 LightGBM 成功率预测模型...
Training until validation scores don't improve for 50 rounds
[50]	training's binary_logloss: 0.276364	training's auc: 0.960269	valid_1's binary_logloss: 0.266719	valid_1's auc: 0.963872
[100]	training's binary_logloss: 0.250691	training's auc: 0.962091	valid_1's binary_logloss: 0.240603	valid_1's auc: 0.965289
[150]	training's binary_logloss: 0.245658	training's auc: 0.963112	valid_1's binary_logloss: 0.235996	valid_1's auc: 0.966041
[200]	training's binary_logloss: 0.243625	training's auc: 0.963628	valid_1's binary_logloss: 0.234523	valid_1's auc: 0.966344
[250]	training's binary_logloss: 0.242377	training's auc: 0.963957	valid_1's binary_logloss: 0.233873	valid_1's auc: 0.966473
[300]	training's binary_logloss: 0.241311	training's auc: 0.964239	valid_1's binary_logloss: 0.233417	valid_1's auc: 0.966565
[350]	training's binary

In [ ]:
import joblib
import os

MODEL_DIR = "./saved_models_xC"
os.makedirs(MODEL_DIR, exist_ok=True)

# 保存分类器和回归器
joblib.dump(bst_model, f"{MODEL_DIR}/xC_Model.pkl")

print("✅ 模型已成功序列化并保存至本地。")

✅ 模型已成功序列化并保存至本地。
